In [38]:
import os
import re
from dotenv import load_dotenv
import requests
import json
from singleRequest import SingleRequest
from agent import Agent
from conversation import ConversationBtwAgents
from utils import *
load_dotenv(override=True)
api_key = os.getenv("OPENROUTER_API_KEY")
sep = os.getenv("SEP")
working_directory = "/Users/szymek/DrawingApp"

In [27]:
#META PROMPT
metaPrompt = load_file("prompts/metaPrompt.txt")
metaPrompt = metaPrompt.replace("{{OPIS_PROJEKTU}}", load_file("projectDescription.txt"))
metaAgent = Agent(model="google/gemini-2.5-flash-lite", typeOfAgent="Meta", api_key=api_key, temperature=1, system_prompt=metaPrompt)


In [28]:
metaAgent.request()

'```json\n{\n    "PM": "# SYSTEM PROMPT: PRODUCT MANAGER (PM)\\n\\nJesteś Product Managerem. Twoim zadaniem jest analiza otrzymanego opisu projektu i przekształcenie go w szczegółową specyfikację funkcjonalną.\\n\\n**Projekt:** \'Program desktopowy do rysowania kształtów geometrycznych na płaszczyźnie. Użytkownik wybiera kształt, podaje wymiary i program go rysuje. Program ma być w stanie zapisać i odtworzyć projekt. Całość kodu jest napisana w języku Python. Program nie powinien korzystać z internetu.\'\\n\\n**Twoje zadania:**\\n1.  **Cel Biznesowy:** Zdefiniuj krótki, zwięzły cel biznesowy projektu.\\n2.  **Persony Użytkowników:** Zdecyduj, kim są główni użytkownicy tego programu.\\n3.  **User Stories:** Wygeneruj listę user stories. Każda user story powinna być jasna, zwięzła i reprezentować konkretną potrzebę użytkownika. Użyj formatu:\\n    *   ID: Unikalny identyfikator (np. 1, 2, 3...)\\n    *   Jako [rola użytkownika]: Kim jest użytkownik.\\n    *   Chcę [funkcjonalność]: Co uż

In [37]:
prompts = metaAgent.get_last_message().get_content()
start = prompts.find("{")
end = prompts.rfind("}")
prompts = prompts[start:end+1]
# prompts = prompts.replace("\n", "")
prompts = json.loads(prompts)
print(prompts['PM'])

# SYSTEM PROMPT: PRODUCT MANAGER (PM)

Jesteś Product Managerem. Twoim zadaniem jest analiza otrzymanego opisu projektu i przekształcenie go w szczegółową specyfikację funkcjonalną.

**Projekt:** 'Program desktopowy do rysowania kształtów geometrycznych na płaszczyźnie. Użytkownik wybiera kształt, podaje wymiary i program go rysuje. Program ma być w stanie zapisać i odtworzyć projekt. Całość kodu jest napisana w języku Python. Program nie powinien korzystać z internetu.'

**Twoje zadania:**
1.  **Cel Biznesowy:** Zdefiniuj krótki, zwięzły cel biznesowy projektu.
2.  **Persony Użytkowników:** Zdecyduj, kim są główni użytkownicy tego programu.
3.  **User Stories:** Wygeneruj listę user stories. Każda user story powinna być jasna, zwięzła i reprezentować konkretną potrzebę użytkownika. Użyj formatu:
    *   ID: Unikalny identyfikator (np. 1, 2, 3...)
    *   Jako [rola użytkownika]: Kim jest użytkownik.
    *   Chcę [funkcjonalność]: Co użytkownik chce zrobić.
    *   Aby [korzyść]: Dlacz

In [39]:
PMAgent = Agent(model="google/gemini-2.5-flash-lite", api_key=api_key, typeOfAgent="Canvas",system_prompt=prompts['PM'])

In [40]:
PMAgent.request()

'Jasne, oto analiza opisu projektu i przekształcenie go w szczegółową specyfikację funkcjonalną:\n\n## Cel Biznesowy\n\nUmożliwienie użytkownikom prostego tworzenia i manipulowania kształtami geometrycznymi w aplikacji desktopowej, która zapewnia podstawowe funkcje projektowania oraz możliwość zapisywania i odtwarzania pracy, bez konieczności połączenia z internetem.\n\n## Persony Użytkowników\n\n*   **Uczeń/Student:** Osoba ucząca się podstaw geometrii lub grafikowania, potrzebująca prostego narzędzia do wizualizacji kształtów i ich wymiarów.\n*   **Hobbysta:** Osoba zainteresowana tworzeniem prostych grafik wektorowych, szkiców technicznych lub wizualizacji koncepcji, która nie potrzebuje zaawansowanych funkcji profesjonalnych programów graficznych.\n*   **Szkoleniowiec/Nauczyciel:** Osoba prowadząca warsztaty lub lekcje z geometrii lub podstaw projektowania, potrzebująca narzędzia do demonstracji kształtów i ich właściwości.\n\n## User Stories\n\n| ID  | Jako | Chcę | Aby |\n| --- |

In [41]:
print(PMAgent.get_last_message().get_content())

Jasne, oto analiza opisu projektu i przekształcenie go w szczegółową specyfikację funkcjonalną:

## Cel Biznesowy

Umożliwienie użytkownikom prostego tworzenia i manipulowania kształtami geometrycznymi w aplikacji desktopowej, która zapewnia podstawowe funkcje projektowania oraz możliwość zapisywania i odtwarzania pracy, bez konieczności połączenia z internetem.

## Persony Użytkowników

*   **Uczeń/Student:** Osoba ucząca się podstaw geometrii lub grafikowania, potrzebująca prostego narzędzia do wizualizacji kształtów i ich wymiarów.
*   **Hobbysta:** Osoba zainteresowana tworzeniem prostych grafik wektorowych, szkiców technicznych lub wizualizacji koncepcji, która nie potrzebuje zaawansowanych funkcji profesjonalnych programów graficznych.
*   **Szkoleniowiec/Nauczyciel:** Osoba prowadząca warsztaty lub lekcje z geometrii lub podstaw projektowania, potrzebująca narzędzia do demonstracji kształtów i ich właściwości.

## User Stories

| ID  | Jako | Chcę | Aby |
| --- | --- | --- | ---

In [45]:
print(prompts['TL'])

# SYSTEM PROMPT: TECH LEAD (TL)

Jesteś Tech Leadem. Twoim zadaniem jest przekształcenie specyfikacji funkcjonalnej (User Stories) i planu technicznego (Architektura) w konkretny zestaw zadań dla zespołu deweloperskiego.

**Wejście (do analizy):**
-   User Stories wygenerowane przez PM.
-   Architektura (Stack Technologiczny, Struktura Katalogów/Plików) wygenerowana przez Architekta.
-   Diagramy stworzone przez Architekta (w celu zrozumienia przepływów).

**Twoje zadania:**
1.  **Rozbicie na Zadania:** Przeanalizuj User Stories i architekturę. Podziel projekt na możliwie jak najmniejsze, atomowe zadania deweloperskie. Każde zadanie powinno być niezależne lub mieć jasno zdefiniowane zależności.
    *   Zadania powinny obejmować implementację poszczególnych funkcjonalności, logiki biznesowej, budowanie UI, obsługę danych, zapis/odczyt.
    *   Zdefiniuj zależności między zadaniami (np. zadanie B zależy od zadania A).

2.  **Definicja Zadań:** Dla każdego zadania podaj:
    *   `id`: Uni

In [43]:
ARCHAgent = Agent(model="google/gemini-2.5-flash-lite", api_key=api_key, typeOfAgent="Canvas",system_prompt=prompts['ARCH'])
ARCHAgent.request(PMAgent.get_last_message().get_content())

'Witaj PM! Fantastyczna inicjatywa! Widzę potencjał w tym prostym, ale użytecznym narzędziu. Analiza User Stories i celów biznesowych jest kluczowa, a ja jako Architekt Oprogramowania, mam za zadanie zapewnić solidne fundamenty technologiczne, które pozwolą na realizację tego projektu w sposób efektywny i skalowalny.\n\nBiorąc pod uwagę kluczowe wymagania - **całość w Pythonie** i **brak dostępu do Internetu**, oto moja propozycja architektoniczna:\n\n---\n\n## 1. Stack Technologiczny\n\nPodstawą naszego projektu będzie **Python**. Jest to wybór strategiczny, umożliwiający nam szybki rozwój i wykorzystanie bogactwa jego ekosystemu, nawet w warunkach offline.\n\n*   **Język Programowania:** **Python 3.x** (zalecam najnowszą stabilną wersję).\n*   **Framework/Biblioteka GUI:**\n    *   **Propozycja:** **PyQt6** (lub PyQt5, jeśli istnieją ograniczenia licencyjne lub zależności w specyficznym środowisku).\n    *   **Uzasadnienie:** PyQt oferuje bogaty zestaw gotowych widżetów, doskonałe ws

In [48]:
TLAgent = Agent(model="google/gemini-2.5-flash-lite", api_key=api_key, typeOfAgent="Canvas",system_prompt=prompts['TL'])
TLAgent.request(PMAgent.get_last_message().get_content()+"\n\n"+ARCHAgent.get_last_message().get_content())

'Jasne, jestem gotów do przełożenia tej solidnej analizy architektonicznej i specyfikacji funkcjonalnej na konkretne zadania deweloperskie. Poniżej znajduje się lista zadań w formacie JSON, uwzględniająca zaproponowaną strukturę projektu i technologię.\n\n```json\n[\n  {\n    "id": 1,\n    "task_name": "Inicjalizacja projektu i konfiguracja środowiska",\n    "description": "Utwórz podstawową strukturę katalogów projektu zgodnie z architekturą. Zainicjalizuj repozytorium Git. Stwórz plik requirements.txt i dodaj podstawowe zależności (np. PyQt6). Skonfiguruj wirtualne środowisko Pythona.",\n    "dependencies": []\n  },\n  {\n    "id": 2,\n    "task_name": "Stworzenie głównego okna aplikacji (MainWindow)",\n    "description": "Zaimplementuj klasę `MainWindow` w `src/app/main_window.py` używając PyQt6. Okno powinno mieć tytuł \'Geometric Designer\', być konfigurowalne pod względem rozmiaru i zawierać podstawowy layoutu, który pomieści płaszczyznę rysowania i panel kontrolek. Nie dodawaj j

In [55]:
tasks = TLAgent.get_last_message().get_content()
start = tasks.find("[")
end = tasks.rfind("]")
tasks = tasks[start:end+1]
tasks = json.loads(tasks)

In [56]:
print(tasks)

[{'id': 1, 'task_name': 'Inicjalizacja projektu i konfiguracja środowiska', 'description': 'Utwórz podstawową strukturę katalogów projektu zgodnie z architekturą. Zainicjalizuj repozytorium Git. Stwórz plik requirements.txt i dodaj podstawowe zależności (np. PyQt6). Skonfiguruj wirtualne środowisko Pythona.', 'dependencies': []}, {'id': 2, 'task_name': 'Stworzenie głównego okna aplikacji (MainWindow)', 'description': "Zaimplementuj klasę `MainWindow` w `src/app/main_window.py` używając PyQt6. Okno powinno mieć tytuł 'Geometric Designer', być konfigurowalne pod względem rozmiaru i zawierać podstawowy layoutu, który pomieści płaszczyznę rysowania i panel kontrolek. Nie dodawaj jeszcze żadnych widżetów do layoutu.", 'dependencies': [1]}, {'id': 3, 'task_name': 'Implementacja punktu wejścia aplikacji', 'description': 'Utwórz plik `src/main.py`. Zaimplementuj funkcję `main()`, która zainicjalizuje aplikację PyQt, utworzy instancję `MainWindow` i uruchomi pętlę zdarzeń aplikacji.', 'dependen

In [ ]:
canvasAgent = Agent(model="google/gemini-2.5-flash-lite", api_key=api_key, typeOfAgent="Canvas")
sentinelAgent = Agent(model="google/gemini-2.5-flash-lite", api_key=api_key, typeOfAgent="Sentinel")
seniorSoftArchAgent = Agent(model="google/gemini-2.5-flash-lite", api_key=api_key, typeOfAgent="SeniorSoftwareArchitect")
techLeadAgent = Agent(model="google/gemini-2.5-flash-lite", api_key=api_key, typeOfAgent="TechLead")
codeScoutAgent = Agent(model="google/gemini-2.5-flash-lite", api_key=api_key, typeOfAgent="CodeScout")
projectingPart = ConversationBtwAgents(canvasAgent, sentinelAgent, initial_message=load_initial_message("initialMessage.txt", "projectDiscription.txt"), first_agent="Canvas")

In [ ]:
#1. GENEROWANIE SPECYFIKACJI
count = 1
while "<ENDOFDOCSGENERATION>" not in sentinelAgent.get_last_message().get_content():
    print(f"{count}. {projectingPart.get_current_agent()}")
    projectingPart.next_request()
    if projectingPart.get_current_agent() == "Canvas":
        print(get_last_sentence(sentinelAgent.get_last_message().get_content()))
    count+=1
export_specs(sentinelAgent.get_last_message().get_content())
specs=sentinelAgent.get_last_message().get_content()

In [ ]:
#2. GENEROWANIE PLANU (ZADAŃ ZALEŻNOŚCI ITP)
plan = seniorSoftArchAgent.request(specs)
taskjson = seniorSoftArchAgent.get_last_message().get_content()
start = taskjson.find('{')
end = taskjson.rfind('}')
taskjson = taskjson[start:end+1].replace("\n","")
tasks = json.loads(taskjson)['tasks']
project_name = json.loads(taskjson)['project_name']
architecture_style = json.loads(taskjson)['architecture_style']
root_directory = json.loads(taskjson)['root_directory']
directory_structure = json.loads(taskjson)['directory_structure']
json.dump(tasks, open("tasks.json", "w", encoding="utf-8"),ensure_ascii=False)


In [ ]:
#3. IMPLEMENTACJA
tasksResults = {}
tasksTemp = tasks.copy()
while len(tasksTemp)!=0:
    for task in tasksTemp:
        if task['id'] not in tasksResults.keys():
            result=''
            if 'dependencies' not in task.keys():
                print(f'{len(tasksTemp)}. {task["id"]}')
                systemPromptFile = 'systemPromptTechLead.txt' if task['agent_type'] == 'B' else 'systemPromptSeniorSoftwareDeveloper.txt'
                request = SingleRequest(initial_message=messageGenerator([],specs, task['instructions'],tasksResults),
                    model="google/gemini-2.5-flash-lite",
                    system_prompt=load_file(systemPromptFile),
                    api_key=api_key,
                    temperature=1.5)
                result = request.next_message()
                tasksResults[task['id']]=result
                tasksTemp.remove(task)
            else:
                SKIPFORNOW=False
                for dep in task['dependencies']:
                    if dep not in tasksResults.keys():
                        SKIPFORNOW=True
                if not SKIPFORNOW:
                    print(f'{len(tasksTemp)}. {task["id"]}')
                    systemPromptFile = 'systemPromptTechLead.txt' if task['agent_type'] == 'B' else 'systemPromptSeniorSoftwareDeveloper.txt'
                    request = SingleRequest(initial_message=messageGenerator(task['dependencies'],specs, task['instructions'],tasksResults),
                        model="google/gemini-2.5-flash-lite",
                        system_prompt=load_file(systemPromptFile),
                        api_key=api_key,
                        temperature=1.5)
                    result = request.next_message()
                    tasksResults[task['id']]=result
                    tasksTemp.remove(task)
            if len(result)>2:
                parse_and_save_files(result,output_dir='.')

In [ ]:
#4. QA TESTS
#5. 

In [ ]:

# PRZYKŁAD UŻYCIA W PIPELINE:
# 1. Agent Scout zwraca JSON:
# scout_response = '{"symbols": ["json.load", "math.sqrt"]}'
# symbols = json.loads(scout_response)['symbols']
import PyQt6.QtWidgets
# 2. Python generuje dokumentację:
real_docs = get_real_docs(["PyQt6.QtWidgets"])
print(real_docs)

In [ ]:
import importlib
import inspect
import types


# TEST
print(get_real_docs(["PyQt6.QtWidgets", "PyQt6.QtWidgets.QPushButton"]))

In [ ]:
import ast

def export_symbols(source):
    tree = ast.parse(source)
    
    imports = {} # Mapuje aliasy na pełne ścieżki, np. 'tk': 'tkinter'
    used_names = set()

    for node in ast.walk(tree):
        # 1. Obsługa: import tkinter as tk
        if isinstance(node, ast.Import):
            for n in node.names:
                imports[n.asname or n.name] = n.name
        
        # 2. Obsługa: from app.core.models import ShapeStyle
        elif isinstance(node, ast.ImportFrom):
            modul = node.module or ""
            for n in node.names:
                full_path = f"{modul}.{n.name}"
                imports[n.asname or n.name] = full_path

        # 3. Obsługa adnotacji typów (np. canvas: tk.Canvas)
        elif isinstance(node, ast.Attribute):
            # Składamy nazwy typu 'tk.Canvas'
            if isinstance(node.value, ast.Name):
                used_names.add(f"{node.value.id}.{node.attr}")
        
        # 4. Obsługa prostych nazw typów (np. ShapeStyle)
        elif isinstance(node, ast.Name):
            # Interesują nas tylko nazwy w kontekście typów lub wywołań
            # (ignorujemy zmienne lokalne w tym uproszczeniu)
            used_names.add(node.id)

    # Łączenie aliasów z pełnymi ścieżkami
    results = set()
    for name in used_names:
        if '.' in name:
            prefiks, reszta = name.split('.', 1)
            if prefiks in imports:
                results.add(f"{imports[prefiks]}.{reszta}")
            else:
                results.add(name)
        else:
            if name in imports:
                results.add(imports[name])
            # Tutaj można by odfiltrować built-iny jak 'float', 'str'
    
    return sorted(list(results))

# Przykład użycia:
kod = """
def funkcja_globalna(a):
    return a * 2

class MojaKlasa:
    def metoda_instancji(self):
        pass
    
    @staticmethod
    def metoda_statyczna():
        print("Cześć")
"""



In [ ]:
with open("app/canvas_renderer.py", "r") as file:
    code = file.read()
    result = export_symbols(code)
    print(result)

In [ ]:
allSymbols = result

In [ ]:
allSymbols

In [ ]:
import importlib
import inspect
import types
import os
import ast

def get_real_docs(symbol_list, project_root="."):
    docs = ""
    
    for symbol in symbol_list:
        try:
            found_locally = False
            # --- KROK 1: POSZUKIWANIE W PLIKACH LOKALNYCH ---
            # Zamieniamy kropki na ścieżkę (np. "my_pkg.utils" -> "my_pkg/utils.py")
            rel_path = symbol.replace('.', os.sep)
            possible_files = [
                os.path.join(project_root, rel_path + ".py"),
                os.path.join(project_root, rel_path, "__init__.py")
            ]

            for file_path in possible_files:
                if os.path.exists(file_path):
                    docs += f"--- LOCAL FILE SOURCE: {file_path} ---\n"
                    docs += extract_docs_from_file(file_path, symbol)
                    found_locally = True
                    break
            
            if found_locally:
                docs += "\n" + "="*30 + "\n"
                continue # Jeśli znaleźliśmy lokalnie, możemy przeskoczyć import (lub usunąć 'continue' by połączyć dane)

            # --- KROK 2: DYNAMICZNY IMPORT (Standardowa logika) ---
            parts = symbol.split('.')
            module_name = parts[0]
            
            try:
                module = importlib.import_module(module_name)
            except ImportError:
                docs += f"--- ERROR: Could not import module {module_name} ---\n"
                continue
            
            obj = module
            for part in parts[1:]:
                obj = getattr(obj, part, None)
                if obj is None:
                    docs += f"--- ERROR: {symbol} not found in {module_name} ---\n"
                    break
            
            if obj is None: continue

            docs += f"--- RUNTIME DOCUMENTATION: {symbol} ---\n"
            
            if isinstance(obj, types.ModuleType):
                classes = [attr for attr in dir(obj) if isinstance(getattr(obj, attr, None), type) and not attr.startswith('_')]
                docs += f"Type: Module\nAvailable Classes: {', '.join(classes[:50])}\n"
            else:
                try:
                    sig = inspect.signature(obj)
                    docs += f"Signature: {symbol}{sig}\n"
                except: pass

                if obj.__doc__:
                    docs += f"Docstring:\n{obj.__doc__[:2000]}\n"
            
            docs += "\n" + "="*30 + "\n"

        except Exception as e:
            docs += f"--- ERROR analyzing {symbol}: {str(e)} ---\n"
    
    return docs

def extract_docs_from_file(file_path, symbol):
    """Przeszukuje AST pliku w poszukiwaniu konkretnego symbolu i jego dokumentacji."""
    try:
        with open(file_path, "r", encoding="utf-8") as f:
            tree = ast.parse(f.read())
        
        # Pobieramy ostatni człon, np. z "projekt.utils.validator" bierzemy "validator"
        target_name = symbol.split('.')[-1]
        
        # Jeśli symbol to po prostu nazwa modułu (plik), zwróć docstring modułu
        if file_path.endswith(f"{target_name}.py") or file_path.endswith("__init__.py"):
            module_doc = ast.get_docstring(tree)
            if module_doc:
                return f"Module Docstring:\n{module_doc}\n"

        # Przeszukujemy definicje wewnątrz pliku
        for node in tree.body:
            # Sprawdzamy klasy i funkcje
            if isinstance(node, (ast.ClassDef, ast.FunctionDef, ast.AsyncFunctionDef)):
                if node.name == target_name:
                    doc = ast.get_docstring(node)
                    type_name = "Class" if isinstance(node, ast.ClassDef) else "Function"
                    return f"Type: {type_name}\nName: {node.name}\nDocstring:\n{doc if doc else 'No docstring found.'}\n"
        
        return f"Symbol '{target_name}' found in file structure but has no docstring or is not a top-level definition.\n"

    except Exception as e:
        return f"Error parsing AST for {symbol}: {e}\n"

In [ ]:
real_docs = get_real_docs(allSymbols)

In [ ]:
print(extract_docs_from_file("app/core/models.py","ShapeStyle"))

In [ ]:
print(real_docs)